In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("../data/raw/Telco-Customer-Churn.csv")

In [3]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df.loc[df["tenure"] == 0, "TotalCharges"] = 0

In [4]:
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"].map({"No": 0, "Yes": 1})

In [5]:
from sklearn.model_selection import train_test_split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [7]:
print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

Training shape: (5634, 19)
Test shape: (1409, 19)


In [8]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


In [9]:
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    column
    for column in X_train.columns
    if column not in numeric_features
]

In [10]:
print("Numerical Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['tenure', 'MonthlyCharges', 'TotalCharges']

Categorical Features:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

In [12]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)


In [14]:
print("Numerical:", numeric_features)
print("Categorical:", categorical_features)

Numerical: ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [15]:
from sklearn.linear_model import LogisticRegression
model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)
model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)

In [16]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[926 109]
 [165 209]]
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



In [17]:
from sklearn.dummy import DummyClassifier
dummy_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DummyClassifier(strategy="most_frequent"))
    ]
)
dummy_pipeline.fit(X_train, y_train)

dummy_pred = dummy_pipeline.predict(X_test)
print(confusion_matrix(y_test,dummy_pred))
print(classification_report(y_test, dummy_pred, zero_division=0))


[[1035    0]
 [ 374    0]]
              precision    recall  f1-score   support

           0       0.73      1.00      0.85      1035
           1       0.00      0.00      0.00       374

    accuracy                           0.73      1409
   macro avg       0.37      0.50      0.42      1409
weighted avg       0.54      0.73      0.62      1409



In [18]:
y_proba = model_pipeline.predict_proba(X_test)[:, 1]
from sklearn.metrics import roc_auc_score, average_precision_score
roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)
threshold = 0.35

y_pred_035 = (y_proba >= threshold).astype(int)
print(confusion_matrix(y_test, y_pred_035))
print(classification_report(y_test, y_pred_035))

ROC-AUC: 0.8420341522643313
PR-AUC: 0.6337024099769349
[[813 222]
 [110 264]]
              precision    recall  f1-score   support

           0       0.88      0.79      0.83      1035
           1       0.54      0.71      0.61       374

    accuracy                           0.76      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.79      0.76      0.77      1409



In [19]:
y_proba = model_pipeline.predict_proba(X_test)[:, 1]
from sklearn.metrics import roc_auc_score, average_precision_score
roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)

ROC-AUC: 0.8420341522643313
PR-AUC: 0.6337024099769349


In [20]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

for threshold in [0.50, 0.40, 0.35, 0.30]:
    preds = (y_proba >= threshold).astype(int)

    print(f"Threshold: {threshold}")
    print("Accuracy :", round(accuracy_score(y_test, preds), 3))
    print("Precision:", round(precision_score(y_test, preds), 3))
    print("Recall   :", round(recall_score(y_test, preds), 3))
    print("F1       :", round(f1_score(y_test, preds), 3))
    print("-" * 30)

Threshold: 0.5
Accuracy : 0.806
Precision: 0.657
Recall   : 0.559
F1       : 0.604
------------------------------
Threshold: 0.4
Accuracy : 0.777
Precision: 0.568
Recall   : 0.668
F1       : 0.614
------------------------------
Threshold: 0.35
Accuracy : 0.764
Precision: 0.543
Recall   : 0.706
F1       : 0.614
------------------------------
Threshold: 0.3
Accuracy : 0.749
Precision: 0.519
Recall   : 0.754
F1       : 0.615
------------------------------


In [21]:
X_subtrain, X_val, y_subtrain, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train
)

In [22]:
from sklearn.base import clone
validation_model = clone(model_pipeline)
validation_model.fit(X_subtrain, y_subtrain)
val_proba = validation_model.predict_proba(X_val)[:, 1]
for threshold in [0.50, 0.45, 0.40, 0.35, 0.30, 0.25]:
    val_pred = (val_proba >= threshold).astype(int)

    print(f"Threshold: {threshold}")
    print("Accuracy :", round(accuracy_score(y_val, val_pred), 3))
    print("Precision:", round(precision_score(y_val, val_pred), 3))
    print("Recall   :", round(recall_score(y_val, val_pred), 3))
    print("F1       :", round(f1_score(y_val, val_pred), 3))
    print("-" * 30)

Threshold: 0.5
Accuracy : 0.803
Precision: 0.658
Recall   : 0.535
F1       : 0.59
------------------------------
Threshold: 0.45
Accuracy : 0.801
Precision: 0.633
Recall   : 0.591
F1       : 0.611
------------------------------
Threshold: 0.4
Accuracy : 0.789
Precision: 0.595
Recall   : 0.644
F1       : 0.619
------------------------------
Threshold: 0.35
Accuracy : 0.779
Precision: 0.567
Recall   : 0.706
F1       : 0.629
------------------------------
Threshold: 0.3
Accuracy : 0.763
Precision: 0.538
Recall   : 0.751
F1       : 0.627
------------------------------
Threshold: 0.25
Accuracy : 0.74
Precision: 0.507
Recall   : 0.802
F1       : 0.621
------------------------------


In [23]:
best_threshold=0.35
final_model=clone(model_pipeline)
final_model.fit(X_train, y_train)
test_proba=final_model.predict_proba(X_test)[:, 1]
final_test_pred = (test_proba >= best_threshold).astype(int)
print(confusion_matrix(y_test, final_test_pred))
print(classification_report(y_test, final_test_pred))

[[813 222]
 [110 264]]
              precision    recall  f1-score   support

           0       0.88      0.79      0.83      1035
           1       0.54      0.71      0.61       374

    accuracy                           0.76      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.79      0.76      0.77      1409



In [24]:
from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}
results = []

for name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring=scoring
    )

    results.append({
        "Model": name,
        "Accuracy Mean": scores["test_accuracy"].mean(),
        "Precision Mean": scores["test_precision"].mean(),
        "Recall Mean": scores["test_recall"].mean(),
        "F1 Mean": scores["test_f1"].mean(),
        "F1 Std": scores["test_f1"].std(),
        "ROC-AUC Mean": scores["test_roc_auc"].mean(),
        "ROC-AUC Std": scores["test_roc_auc"].std()
    })
    results_df = pd.DataFrame(results)
results_df

,Model,Accuracy Mean,Precision Mean,Recall Mean,F1 Mean,F1 Std,ROC-AUC Mean,ROC-AUC Std
0,Logistic Regression,0.804400,0.658236,0.548495,0.598208,0.027205,0.845684,0.013811
1,Decision Tree,0.717250,0.468676,0.487625,0.477804,0.012925,0.644592,0.008557
2,Random Forest,0.785586,0.624672,0.482943,0.544539,0.021202,0.822362,0.012180
3,Gradient Boosting,0.802093,0.658974,0.527759,0.585805,0.029643,0.846819,0.013246


In [25]:
from sklearn.model_selection import GridSearchCV
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)
logistic_param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"]
}
logistic_grid = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)
logistic_grid.fit(X_train, y_train)
print("Best parameters:", logistic_grid.best_params_)
print("Best CV F1:", logistic_grid.best_score_)

Best parameters: {'model__C': 0.1, 'model__class_weight': 'balanced'}
Best CV F1: 0.6326202316817229


In [26]:
gb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", GradientBoostingClassifier(random_state=42))
    ]
)
gb_param_grid = {
    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [2, 3]
}
gb_grid = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=gb_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)
gb_grid.fit(X_train, y_train)
print("Best parameters:", gb_grid.best_params_)
print("Best CV F1:", gb_grid.best_score_)

Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': 2, 'model__n_estimators': 100}
Best CV F1: 0.5906472401541496


In [27]:
from sklearn.base import clone

best_logistic = logistic_grid.best_estimator_
tuned_validation_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                C=0.1,
                class_weight="balanced",
                max_iter=1000
            )
        )
    ]
)
tuned_validation_model.fit(X_subtrain, y_subtrain)
tuned_val_proba = tuned_validation_model.predict_proba(X_val)[:, 1]
for threshold in [0.60, 0.55, 0.50, 0.45, 0.40, 0.35, 0.30]:
    val_pred = (tuned_val_proba >= threshold).astype(int)

    print(f"Threshold: {threshold}")
    print("Accuracy :", round(accuracy_score(y_val, val_pred), 3))
    print("Precision:", round(precision_score(y_val, val_pred), 3))
    print("Recall   :", round(recall_score(y_val, val_pred), 3))
    print("F1       :", round(f1_score(y_val, val_pred), 3))
    print("-" * 30)

Threshold: 0.6
Accuracy : 0.781
Precision: 0.573
Recall   : 0.69
F1       : 0.626
------------------------------
Threshold: 0.55
Accuracy : 0.764
Precision: 0.541
Recall   : 0.746
F1       : 0.627
------------------------------
Threshold: 0.5
Accuracy : 0.748
Precision: 0.517
Recall   : 0.791
F1       : 0.625
------------------------------
Threshold: 0.45
Accuracy : 0.727
Precision: 0.491
Recall   : 0.818
F1       : 0.614
------------------------------
Threshold: 0.4
Accuracy : 0.7
Precision: 0.464
Recall   : 0.85
F1       : 0.601
------------------------------
Threshold: 0.35
Accuracy : 0.661
Precision: 0.432
Recall   : 0.88
F1       : 0.579
------------------------------
Threshold: 0.3
Accuracy : 0.639
Precision: 0.417
Recall   : 0.904
F1       : 0.571
------------------------------


In [28]:
best_threshold=0.55
final_model = logistic_grid.best_estimator_
test_proba = final_model.predict_proba(X_test)[:, 1]
final_pred = (test_proba >= best_threshold).astype(int)
print(confusion_matrix(y_test, final_pred))
print(classification_report(y_test, final_pred))
print("ROC-AUC:", roc_auc_score(y_test, test_proba))
print("PR-AUC:", average_precision_score(y_test, test_proba))

[[781 254]
 [ 93 281]]
              precision    recall  f1-score   support

           0       0.89      0.75      0.82      1035
           1       0.53      0.75      0.62       374

    accuracy                           0.75      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.80      0.75      0.77      1409

ROC-AUC: 0.8408302978635459
PR-AUC: 0.6330342971947791


In [29]:
feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
feature_names[:20]

array(['num__tenure', 'num__MonthlyCharges', 'num__TotalCharges',
       'cat__gender_Female', 'cat__gender_Male', 'cat__SeniorCitizen_0',
       'cat__SeniorCitizen_1', 'cat__Partner_No', 'cat__Partner_Yes',
       'cat__Dependents_No', 'cat__Dependents_Yes',
       'cat__PhoneService_No', 'cat__PhoneService_Yes',
       'cat__MultipleLines_No', 'cat__MultipleLines_No phone service',
       'cat__MultipleLines_Yes', 'cat__InternetService_DSL',
       'cat__InternetService_Fiber optic', 'cat__InternetService_No',
       'cat__OnlineSecurity_No'], dtype=object)

In [30]:
logistic_model = final_model.named_steps["model"]
coefficients = logistic_model.coef_[0]
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})
feature_importance = feature_importance.sort_values(
    by="Coefficient",
    ascending=False
)
feature_importance.tail(10)

,Feature,Coefficient
10,cat__Dependents_Yes,-0.116483
42,cat__PaymentMethod_Bank transfer (automatic),-0.117337
43,cat__PaymentMethod_Credit card (automatic),-0.119937
30,cat__TechSupport_Yes,-0.142626
40,cat__PaperlessBilling_No,-0.168428
13,cat__MultipleLines_No,-0.171734
21,cat__OnlineSecurity_Yes,-0.175549
16,cat__InternetService_DSL,-0.321830
39,cat__Contract_Two year,-0.651307
0,num__tenure,-0.971786


In [31]:
feature_importance.head(10)

,Feature,Coefficient
37,cat__Contract_Month-to-month,0.684518
17,cat__InternetService_Fiber optic,0.399905
2,num__TotalCharges,0.284917
44,cat__PaymentMethod_Electronic check,0.283824
19,cat__OnlineSecurity_No,0.253625
28,cat__TechSupport_No,0.220702
41,cat__PaperlessBilling_Yes,0.165699
36,cat__StreamingMovies_Yes,0.151540
33,cat__StreamingTV_Yes,0.140219
22,cat__OnlineBackup_No,0.121366


In [32]:
error_analysis = X_test.copy()

error_analysis["Actual"] = y_test
error_analysis["Predicted"] = final_pred
error_analysis["ChurnProbability"] = test_proba
false_negatives = error_analysis[
    (error_analysis["Actual"] == 1) &
    (error_analysis["Predicted"] == 0)
]
false_negatives["Contract"].value_counts(normalize=True)
false_negatives["InternetService"].value_counts(normalize=True)
false_negatives["TechSupport"].value_counts(normalize=True)

TechSupport
No                     0.387097
Yes                    0.344086
No internet service    0.268817
Name: proportion, dtype: float64

In [33]:

false_negatives.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Actual,Predicted,ChurnProbability
2488,Male,0,No,No,31,Yes,No,DSL,No,No,...,No,No,Month-to-month,Yes,Electronic check,55.25,1715.65,1,0,0.417251
5086,Male,0,No,No,45,Yes,No,No,No internet service,No internet service,...,No internet service,No internet service,One year,Yes,Electronic check,20.40,930.45,1,0,0.102273
3408,Female,0,No,No,4,Yes,No,DSL,No,Yes,...,No,No,Month-to-month,No,Credit card (automatic),50.70,151.30,1,0,0.501802
7011,Female,0,No,No,4,Yes,No,DSL,Yes,Yes,...,No,No,Month-to-month,Yes,Mailed check,60.40,272.15,1,0,0.417325
3158,Male,0,Yes,No,18,Yes,No,DSL,No,No,...,No,No,Month-to-month,Yes,Mailed check,49.55,878.35,1,0,0.523748


In [34]:
false_negatives.shape

(93, 22)

In [35]:
false_negatives[
    ["tenure", "MonthlyCharges", "TotalCharges", "ChurnProbability"]
].describe()

,tenure,MonthlyCharges,TotalCharges,ChurnProbability
count,93.000000,93.000000,93.000000,93.000000
mean,28.602151,60.980108,2376.841398,0.360317
std,24.409611,32.158283,2537.617158,0.130745
min,1.000000,18.950000,19.300000,0.019044
25%,4.000000,20.950000,146.300000,0.264910
50%,24.000000,56.700000,1544.050000,0.379208
75%,53.000000,94.100000,4062.200000,0.460300
max,72.000000,115.650000,7968.850000,0.549847


In [36]:
actual_churn = error_analysis[
    error_analysis["Actual"] == 1
]
actual_churn.shape

(374, 22)

In [37]:
print("False Negatives:")
print(false_negatives["Contract"].value_counts(normalize=True))

print("\nAll Actual Churn:")
print(actual_churn["Contract"].value_counts(normalize=True))

False Negatives:
Contract
Month-to-month    0.559140
One year          0.344086
Two year          0.096774
Name: proportion, dtype: float64

All Actual Churn:
Contract
Month-to-month    0.879679
One year          0.096257
Two year          0.024064
Name: proportion, dtype: float64


In [38]:
print("False Negatives:")
print(false_negatives["InternetService"].value_counts(normalize=True))

print("\nAll Actual Churn:")
print(actual_churn["InternetService"].value_counts(normalize=True))

False Negatives:
InternetService
DSL            0.419355
Fiber optic    0.311828
No             0.268817
Name: proportion, dtype: float64

All Actual Churn:
InternetService
Fiber optic    0.673797
DSL            0.259358
No             0.066845
Name: proportion, dtype: float64


In [39]:
print("False Negatives:")
print(false_negatives["TechSupport"].value_counts(normalize=True))

print("\nAll Actual Churn:")
print(actual_churn["TechSupport"].value_counts(normalize=True))

False Negatives:
TechSupport
No                     0.387097
Yes                    0.344086
No internet service    0.268817
Name: proportion, dtype: float64

All Actual Churn:
TechSupport
No                     0.772727
Yes                    0.160428
No internet service    0.066845
Name: proportion, dtype: float64


In [40]:
false_positives = error_analysis[
    (error_analysis["Actual"] == 0) &
    (error_analysis["Predicted"] == 1)
]
false_positives.shape

(254, 22)

In [41]:
false_positives[
    ["tenure", "MonthlyCharges", "TotalCharges", "ChurnProbability"]
].describe()

,tenure,MonthlyCharges,TotalCharges,ChurnProbability
count,254.000000,254.000000,254.000000,254.000000
mean,19.082677,77.422244,1649.882087,0.719209
std,16.194136,20.211187,1620.223303,0.096297
min,1.000000,24.200000,24.200000,0.550752
25%,6.000000,68.762500,402.400000,0.648270
50%,16.000000,79.975000,1054.175000,0.712249
75%,27.500000,93.225000,2392.025000,0.798413
max,67.000000,114.100000,7132.150000,0.918956


In [42]:
actual_no_churn = error_analysis[
    error_analysis["Actual"] == 0
]
actual_no_churn.shape

(1035, 22)

In [43]:
print("False Positives:")
print(false_positives["Contract"].value_counts(normalize=True))

print("\nAll Actual No Churn:")
print(actual_no_churn["Contract"].value_counts(normalize=True))

False Positives:
Contract
Month-to-month    0.956693
One year          0.043307
Name: proportion, dtype: float64

All Actual No Churn:
Contract
Month-to-month    0.428986
Two year          0.315942
One year          0.255072
Name: proportion, dtype: float64


In [44]:
print("False Positives:")
print(false_positives["InternetService"].value_counts(normalize=True))

print("\nAll Actual No Churn:")
print(actual_no_churn["InternetService"].value_counts(normalize=True))

False Positives:
InternetService
Fiber optic    0.728346
DSL            0.271654
Name: proportion, dtype: float64

All Actual No Churn:
InternetService
DSL            0.373913
Fiber optic    0.348792
No             0.277295
Name: proportion, dtype: float64


In [45]:
print("False Positives:")
print(false_positives["TechSupport"].value_counts(normalize=True))

print("\nAll Actual No Churn:")
print(actual_no_churn["TechSupport"].value_counts(normalize=True))

False Positives:
TechSupport
No     0.791339
Yes    0.208661
Name: proportion, dtype: float64

All Actual No Churn:
TechSupport
No                     0.399034
Yes                    0.323671
No internet service    0.277295
Name: proportion, dtype: float64


In [46]:
import joblib
model_bundle = {
    "model": final_model,
    "threshold": best_threshold
}
joblib.dump(
    model_bundle,
    "../models/churn_model.joblib"
)

['../models/churn_model.joblib']

## Error Analysis Findings

- False negatives are disproportionately associated with traditionally lower-risk profiles, such as longer contracts, DSL/no internet service, and having technical support.
- False positives are strongly concentrated among high-risk profiles, especially month-to-month contracts, fiber-optic customers, and customers without technical support.
- This suggests that the model captures the dominant churn patterns well, but struggles with customers whose actual behavior differs from the typical patterns learned from the data.
- These findings describe model behavior and associations; they do not establish causal relationships.

In [47]:
loaded_bundle = joblib.load(
    "../models/churn_model.joblib"
)
loaded_model = loaded_bundle["model"]
loaded_threshold = loaded_bundle["threshold"]
print("Loaded threshold:", loaded_threshold)

Loaded threshold: 0.55


In [48]:
sample = X_test.iloc[[0]]
sample_probability = loaded_model.predict_proba(sample)[:, 1][0]
sample_prediction = int(
    sample_probability >= loaded_threshold
)

print("Churn probability:", sample_probability)
print("Prediction:", sample_prediction)

Churn probability: 0.10917556164702244
Prediction: 0
